# Lattice vertex algebras

`lattice_vertex_algebra.py` constructs the vertex algebra $V_L$ of an integral lattice $L$ [Bor1986, Kac1998]. It is built from two kinds of fields:

* the **Heisenberg currents** $J_0, \dots, J_{n-1}$, one for each basis vector $x_0, \dots, x_{n-1}$ of the lattice (free bosons);
* the **lattice currents** (vertex operators) $e^q$, one for each lattice vector $q$; $e^0$ is the vacuum $\mathbf 1$.

As a vector space $V_L$ is spanned by the normally ordered monomials ${:}\partial^{m_1}J_{i_1} \cdots \partial^{m_k}J_{i_k}\, e^q{:}$, and these form a basis. Lattice vertex algebras include the simple quotients of simply-laced affine vertex algebras at level 1 [FK1980, Seg1981] (sections 1 and 3), free fermions (section 4), and, for indefinite lattices, free field realisations such as the βγ system [FMS1986] (section 6).

The data are:

* a coefficient ring $R$ containing $\mathbb Q$, for example `QQ` or a fraction field such as $\mathbb Q(a)$;
* a **Gram matrix** $B$: symmetric, with integer entries and nonzero determinant, of any signature. The lattice is $\mathbb Z^n$ with the basis $x_0, \dots, x_{n-1}$ and the bilinear form $B(q, p) = \sum_{i,j} q_i B_{ij} p_j$. A lattice vector $q$ is given by its coordinates, as a tuple: `(1, 0)`, or `(1,)` in rank 1;
* optionally a **cocycle** $\varepsilon$, a sign attached to each pair of lattice vectors. The default is the right choice for most purposes; section 3 explains what it is for.

`LatticeVertexAlgebra(R, B)`, also available as `va.Lattice(R, B)`, returns the algebra. `V.J(i)` is $J_i$, `V.e(q)` is $e^q$, and `V.gens()` lists $J_i$ and $e^{\pm x_i}$, which generate $V_L$.

**Conventions** [Kac1998, §5.4–5.5]. Products are written as in `01_getting_started.ipynb`: `a.ope(b)` is the list of poles $[a_{(0)}b, a_{(1)}b, \dots]$, `a.nproduct(b, n)` is $a_{(n)}b$ for any integer $n$, `a.no(b)` is ${:}ab{:} = a_{(-1)}b$. In this notation:

* $J_i(z)J_j(w) \sim \dfrac{B_{ij}}{(z-w)^2}$ and $J_i(z)e^q(w) \sim \dfrac{B(x_i, q)\, e^q(w)}{z-w}$;
* $e^q(z)e^p(w)$ starts at the power $(z-w)^{B(q,p)}$: it has $-B(q,p)$ poles when $B(q, p) < 0$ and is regular otherwise. Precisely, $e^q_{(n)}e^p = \varepsilon(q,p)\, S_{-n-1-B(q,p)}(q)\, e^{q+p}$, where $S_k(q)$ is the complete symmetric function $h_k$ with the power sums $p_m$ replaced by the modes $q(-m)$ of $J_q = \sum_i q_i J_i$, that is $\sum_k S_k(q) z^k = \exp\big(\sum_{m>0} q(-m) z^m/m\big)$ (the *elementary Schur polynomials* of the KP and lattice vertex algebra literature). As states these are normally ordered polynomials in the derivatives of $J_q$ (for instance $S_0 = 1$, $S_1(q) = J_q$, $S_2(q) = \frac12({:}J_qJ_q{:} + \partial J_q)$);
* $\partial e^q = \sum_i q_i\, {:}J_i e^q{:}$;
* the parity of $e^q$ is $B(q, q) \bmod 2$, and the $J_i$ are even. A lattice with some odd $B_{ii}$ (an *odd* lattice) gives a vertex superalgebra (section 4).

**Conformal vectors.** None is chosen by default. The usual one is the conformal vector of the Heisenberg currents, $T_L = \frac12 \sum_{i,j} (B^{-1})_{ij}\, {:}J_iJ_j{:}$, of central charge $n$ (the rank), for which $e^q$ has weight $B(q,q)/2$. For the root lattice of a simply-laced Lie algebra $\mathfrak g$, $V_L$ is the affine vertex algebra $L_1(\mathfrak g)$ and $T_L$ is the Sugawara vector of its currents (sections 2 and 3); for other lattices there is no such affine algebra, and $T_L$ is simply the free boson conformal vector. Adding a background charge, $T = T_L + \partial J_\alpha$, changes the weights and the central charge. A lattice vertex algebra has no names for its conformal vectors: the conformal vector is written as an element and passed to `V.with_conformal_vector(T)` (section 2).

**Relation to Sage.** A Sage `IntegralLattice` can be given in place of its Gram matrix (section 8).

**Prerequisites:** `01_getting_started.ipynb`. Sections 4, 6 and 7 compare with the ghost systems of `04_ghost_systems.ipynb` and use morphisms `V.hom(images, W)`, as in its section 11.

**Elsewhere:** the simple modules $V_{\lambda+L}$ of lattice vertex algebras are in `10_modules.ipynb` (section 2), their Zhu algebras in `09_zhu_algebras.ipynb` (section 8), their fusion products in `11_fusion_and_extensions.ipynb`, and a research example in `14_triplet_algebra.ipynb`.

**Contents**

1. A first example: the $A_1$ lattice
2. Conformal vectors, weights and characters
3. The cocycle: the $A_2$ lattice
4. Odd lattices: free fermions
5. Indefinite lattices: $I_{1,1}$
6. The βγ system inside $V_{I_{1,1}}$
7. Tensor products
8. Unique representation, pickling and Sage lattices
9. Limitations

In [1]:
import vertex_algebras as va
from vertex_algebras import *

## 1. A first example: the $A_1$ lattice

The root lattice of $\mathfrak{sl}_2$ is $\mathbb Z x_0$ with $B(x_0, x_0) = 2$, so its Gram matrix is the $1 \times 1$ matrix $(2)$. The algebra has one Heisenberg current `J0` and the lattice currents $e^{k x_0}$, printed `e^(k)`. The cell below prints the algebra, the names of its generators, and its conformal vector, which is `None` because none has been chosen.

In [2]:
A1 = LatticeVertexAlgebra(QQ, matrix([[2]]))
print(A1)
print(A1.variable_names(), A1.conformal_vector())

Lattice vertex algebra of rank 1 with Gram matrix [2] over Rational Field
('J0', 'e^(1)', 'e^(-1)') None


The fields $H = J_0$, $E = e^{x_0}$ and $F = e^{-x_0}$ satisfy the OPEs of affine $\mathfrak{sl}_2$ at level 1 [FK1980, Seg1981]:

$$H(z)H(w) \sim \frac{2}{(z-w)^2}, \quad H(z)E(w) \sim \frac{2E(w)}{z-w}, \quad H(z)F(w) \sim \frac{-2F(w)}{z-w}, \quad E(z)F(w) \sim \frac{1}{(z-w)^2} + \frac{H(w)}{z-w}.$$

The cell below prints these OPEs. Since $B(x_0, x_0) = 2 \ge 0$, the OPE $E(z)E(w)$ is regular: it starts at $(z-w)^2$, so the first nonzero product is $E_{(-3)}E = e^{2x_0}$. The last line shows the derivative $\partial E = {:}J_0 E{:}$. (A lattice vector of rank 1 is a tuple with one entry, written `(1,)` with a comma.)

In [3]:
H, E, F = A1.J(0), A1.e((1,)), A1.e((-1,))
print('H H :', H.ope(H))
print('H E :', H.ope(E), '   H F :', H.ope(F))
print('E F :', E.ope(F))
print('E E :', E.ope(E), '   E_(-3) E =', E.nproduct(E, -3))
print('dE  =', E.derivative())

H H : [0, 2]
H E : [2*e^(1)]    H F : [-2*e^(-1)]
E F : [J0, 1]
E E : []    E_(-3) E = e^(2)
dE  = :J0 e^(1):


## 2. Conformal vectors, weights and characters

The conformal vector of the Heisenberg currents, $T_L = \frac12 \sum_{i,j} (B^{-1})_{ij}\, {:}J_iJ_j{:}$, makes sense for every lattice. It is written once below, as the function `lattice_conformal_vector`, and used throughout the notebook. For $A_1$ it is $T_L = \frac14 {:}J_0J_0{:}$.

In [4]:
def lattice_conformal_vector(V):
    # T_L = 1/2 sum_ij (B^-1)_ij :J_i J_j: of a lattice vertex algebra V (central charge = rank)
    Binv = V.gram_matrix().inverse()
    J = V.heisenberg_currents()
    return sum(Binv[i, j] / 2 * J[i].no(J[j]) for i in range(V.rank()) for j in range(V.rank()))


lattice_conformal_vector(A1)

1/4*:J0 J0:

**The Sugawara vector of $\mathfrak{sl}_2$.** Section 1 found the currents $H, E, F$ of affine $\mathfrak{sl}_2$ at level $k = 1$ inside $V_{A_1}$. The Sugawara construction applied to these currents gives the conformal vector

$$T_{\mathrm{sug}} = \frac{1}{2(k + h^\vee)} \Big(\tfrac12 {:}HH{:} + {:}EF{:} + {:}FE{:}\Big), \qquad k = 1,\ h^\vee = 2,$$

the sum over a basis of $\mathfrak{sl}_2$ and its dual basis for the normalised invariant form. The function `affine_sugawara` below does this for any list of currents $x_a$ of an affine algebra of level $k$ and dual Coxeter number $h^\vee$: it reads the form off the second-order poles, $x_a(z)x_b(w) \sim G_{ab}/(z-w)^2 + \cdots$ (so $G_{ab} = k(x_a|x_b)$), and returns $\frac{k}{2(k+h^\vee)} \sum_{a,b} (G^{-1})_{ab}\, {:}x_ax_b{:}$. The cell below checks that for $H, E, F$ it is $T_L$: on the root lattice, the Sugawara vector of the affine algebra is the conformal vector of the Heisenberg current [FK1980, Kac1998].

In [5]:
def affine_sugawara(currents, k, h_dual):
    V = currents[0].parent()
    vac = V.one().leading_support()
    G = matrix(V.base_ring(), [[x.nproduct(y, 1).monomial_coefficients().get(vac, 0) for y in currents]
                               for x in currents])
    Gi = G.inverse()
    n = len(currents)
    return k / (2 * (k + h_dual)) * sum(Gi[a, b] * currents[a].no(currents[b]) for a in range(n) for b in range(n))


T_sug = affine_sugawara([H, E, F], 1, 2)
print('T_sug =', T_sug, '   T_sug == T_L:', T_sug == lattice_conformal_vector(A1))

T_sug = 1/4*:J0 J0:    T_sug == T_L: True


`V.with_conformal_vector(T)` returns the same algebra with $T$ as its conformal vector, after checking that $T$ is one (see `01_getting_started.ipynb`, section 4); its elements then have conformal weights (`x.degree()`). With $T_L$ the central charge is 1, and $e^{k x_0}$ has weight $B(kx_0, kx_0)/2 = k^2$. In particular $J_0$ and $e^{\pm x_0}$ all have weight 1, as the currents of an affine algebra should. The cell below prints the central charge, the weights of the generators and those of $e^{kx_0}$ for $k = -2, \dots, 2$.

In [6]:
VA1 = A1.with_conformal_vector(lattice_conformal_vector(A1))
print('c =', VA1.central_charge())
print(VA1.degrees())
print([(k, VA1.e((k,)).degree()) for k in range(-2, 3)])

c = 1
{'J0': 1, 'e^(1)': 1, 'e^(-1)': 1}
[(-2, 4), (-1, 1), (0, 0), (1, 1), (2, 4)]


With conformal weights, `V.weight_space(n)` and `V.character(N)` are available (`01_getting_started.ipynb`, section 5). For a positive definite lattice every weight space is finite-dimensional, and the character is the theta function of the lattice divided by the character of the Heisenberg Fock spaces:

$$\operatorname{ch} V_{A_1} = \frac{\sum_{k \in \mathbb Z} q^{k^2}}{\prod_{m \ge 1} (1 - q^m)}.$$

The cell below prints the character computed by the library and the series on the right, and the basis of the weight-1 space: the three currents $e^{-x_0}$, $e^{x_0}$, $J_0$ of $\mathfrak{sl}_2$.

In [7]:
print(VA1.character(10))
R.<q> = PowerSeriesRing(QQ, default_prec=10)
print((sum(q^(k^2) for k in range(-3, 4)) / prod(1 - q^m for m in range(1, 10))).add_bigoh(10))
print(VA1.weight_space(1).basis())

1 + 3*q + 4*q^2 + 7*q^3 + 13*q^4 + 19*q^5 + 29*q^6 + 43*q^7 + 62*q^8 + 90*q^9 + O(q^10)
1 + 3*q + 4*q^2 + 7*q^3 + 13*q^4 + 19*q^5 + 29*q^6 + 43*q^7 + 62*q^8 + 90*q^9 + O(q^10)
[e^(-1), e^(1), J0]


**Background charge.** For a vector $\alpha$ (with coefficients in $R$), $T = T_L + \partial J_\alpha$, where $J_\alpha = \sum_i \alpha_i J_i$, is again a conformal vector. Its central charge is $n - 12 B(\alpha, \alpha)$, and $e^q$ has weight $B(q,q)/2 - B(\alpha, q)$, while the $J_i$ keep weight 1 [Kac1998].

The cell below takes $\alpha = a x_0$ with a parameter $a$, so the coefficient ring is the fraction field $\mathbb Q(a)$. Then $B(\alpha, \alpha) = 2a^2$, so $c = 1 - 24a^2$, and $e^{\pm x_0}$ have weights $1 \mp 2a$.

In [8]:
K.<a> = Frac(QQ['a'])
AK = LatticeVertexAlgebra(K, matrix([[2]]))
Va = AK.with_conformal_vector(lattice_conformal_vector(AK) + a * AK.J(0).derivative())
print('c =', Va.central_charge())
print(Va.degrees())

c = -24*a^2 + 1
{'J0': 1, 'e^(1)': -2*a + 1, 'e^(-1)': 2*a + 1}


Conformal weights can also be given without a conformal vector, with `degrees=` (a dict on the names of the generators). Not every choice gives a grading: $J_i$ must have weight 1 and the weights of $e^{x_i}$ and $e^{-x_i}$ must add up to $B_{ii}$. Such weights are always those of $T_L + \partial J_\alpha$ for some $\alpha$.

The cell below gives $e^{x_0}$ weight 0 and $e^{-x_0}$ weight 2 (that is, $\alpha = \frac12 x_0$), and then shows two refusals: weights that do not add up to $B_{00} = 2$, and the element $\frac12{:}J_0J_0{:}$, which is $T_L$ for the lattice with $B = (1)$ but not a conformal vector of this one.

In [9]:
Vd = LatticeVertexAlgebra(QQ, matrix([[2]]), degrees={'J0': 1, 'e^(1)': 0, 'e^(-1)': 2})
print(Vd.degrees(), Vd.conformal_vector())
try:
    LatticeVertexAlgebra(QQ, matrix([[2]]), degrees={'J0': 1, 'e^(1)': 1, 'e^(-1)': 2})
except ValueError as err:
    print('ValueError:', err)
try:
    A1.with_conformal_vector(H.no(H) / 2)
except ValueError as err:
    print('ValueError:', err)

{'J0': 1, 'e^(1)': 0, 'e^(-1)': 2} None
ValueError: the weights 1 of e^(x_0) and 2 of e^(-x_0) (diagonal of T_(1)) do not add up to B_ii = 2, so they do not define a grading
ValueError: T is not a Virasoro field: T(z)T(w) has poles [2*:dJ0 J0:, 2*:J0 J0:, 0, 2] instead of [∂T, 2T, 0, c/2]


## 3. The cocycle: the $A_2$ lattice

Why a cocycle is needed: for even lattice vectors $q$, $p$, locality requires $e^q(z)e^p(w) = e^p(w)e^q(z)$, while the vertex operators $e^q(z)$ built from the Heisenberg algebra alone commute only up to the sign $(-1)^{B(q,p)}$. The cocycle corrects this sign: each product $e^q_{(n)}e^p$ is multiplied by a sign $\varepsilon(q, p)$, chosen so that

$$\varepsilon(q,p)/\varepsilon(p,q) = (-1)^{B(q,p) + B(q,q)B(p,p)}$$

[Kac1998, §5.4–5.5]. The default, `epsilon='standard'`, is the bimultiplicative cocycle with $\varepsilon(x_i, x_j) = (-1)^{B_{ij} + B_{ii}B_{jj}}$ for $i > j$ and $1$ for $i \le j$; it always satisfies this condition. Different valid cocycles give isomorphic vertex algebras.

For the root lattice $A_2$ of $\mathfrak{sl}_3$, $B(x_0, x_1) = -1$, so $\varepsilon(x_1, x_0) = -1$. The cell below prints the cocycle on the basis vectors, the resulting OPEs $e^{x_0}(z)e^{x_1}(w)$ and $e^{x_1}(z)e^{x_0}(w)$, which differ by a sign as skew-symmetry requires, and the OPE of $e^{x_0 + x_1}$ with $e^{-x_0-x_1}$, whose sign also comes from the cocycle.

In [10]:
A2 = LatticeVertexAlgebra(QQ, matrix([[2, -1], [-1, 2]]))
a, b = A2.e((1, 0)), A2.e((0, 1))
print('ε(x0,x1) =', A2.cocycle((1, 0), (0, 1)), '   ε(x1,x0) =', A2.cocycle((0, 1), (1, 0)))
print('a b :', a.ope(b), '   b a :', b.ope(a))
print('e^(1,1) e^(-1,-1) :', A2.e((1, 1)).ope(A2.e((-1, -1))))

ε(x0,x1) = 1    ε(x1,x0) = -1
a b : [e^(1,1)]    b a : [-e^(1,1)]
e^(1,1) e^(-1,-1) : [-J0 - J1, -1]


With $T_L$, the weight-1 space is spanned by the two currents and the six $e^q$ with $B(q, q) = 2$, the roots of $\mathfrak{sl}_3$: it has dimension $8 = \dim \mathfrak{sl}_3$, and $V_{A_2}$ is the simple affine vertex algebra $L_1(\mathfrak{sl}_3)$ [FK1980]. As for $\mathfrak{sl}_2$ in section 2, $T_L$ is the Sugawara vector of these eight currents at level $k = 1$ ($h^\vee = 3$), as the last line of the cell below checks with `affine_sugawara`. The central charge is the rank, 2.

In [11]:
VA2 = A2.with_conformal_vector(lattice_conformal_vector(A2))
currents = VA2.weight_space(1).basis()
print(VA2.central_charge(), len(currents))
print(currents)
print(affine_sugawara(currents, 1, 3) == VA2.conformal_vector())

2 8
[e^(-1,-1), e^(-1,0), e^(0,-1), e^(0,1), e^(1,0), e^(1,1), J0, J1]
True


**The trivial cocycle.** `epsilon='trivial'` sets $\varepsilon = 1$. For $A_2$ this violates the condition above, and the constructor issues a `RuntimeWarning`: the algebra it returns is not local.

Every product is computed by `nproduct`, so the axioms can be tested directly. The function `skew_symmetric` below checks the skew-symmetry identity

$$b_{(n)}a = (-1)^{|a||b|} \sum_{j \ge 0} \frac{(-1)^{n+j+1}}{j!}\, \partial^j\big(a_{(n+j)}b\big)$$

for a range of $n$ ($|a|$ is the parity of $a$). The cell prints the warning, and then the OPEs of $e^{x_0}$ and $e^{x_1}$ for the trivial and the standard cocycle, with the result of the check: with the trivial cocycle the two OPEs have the same sign, and skew-symmetry fails.

In [12]:
import warnings


def skew_symmetric(a, b, ns=range(-3, 3), jmax=8):
    s = (-1)^(a.parity() * b.parity())
    return all(b.nproduct(a, n) == s * sum((-1)^(n + j + 1) / factorial(j) * a.nproduct(b, n + j).derivative(j)
                                           for j in range(jmax))
               for n in ns)


with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    A2triv = LatticeVertexAlgebra(QQ, matrix([[2, -1], [-1, 2]]), epsilon='trivial')
print(w[0].category.__name__ + ':', w[0].message)
at, bt = A2triv.e((1, 0)), A2triv.e((0, 1))
print('trivial :', at.ope(bt), bt.ope(at), '  skew-symmetric:', skew_symmetric(at, bt))
print('standard:', a.ope(b), b.ope(a), '  skew-symmetric:', skew_symmetric(a, b))

trivial : [e^(1,1)] [e^(1,1)]   skew-symmetric: False
standard: [e^(1,1)] [-e^(1,1)]   skew-symmetric: True


**A cocycle as a function.** `epsilon=` also takes a function of two lattice vectors (tuples of integers) returning a unit of $R$. The library uses it as it is: it does not check the cocycle identity or the condition above (see section 9).

The standard cocycle of $A_2$ is $\varepsilon(q, p) = (-1)^{q_1 p_0}$. Written as a function it gives the same values, and so the same products, but the algebra is a different object, because its data differ. The cell below checks the values on a few vectors and repeats one OPE.

In [13]:
def eps_A2(q, p):
    return (-1)^(q[1] * p[0])


A2f = LatticeVertexAlgebra(QQ, matrix([[2, -1], [-1, 2]]), epsilon=eps_A2)
vs = [(1, 0), (0, 1), (1, 1), (2, -1), (-1, 3)]
print(A2f is A2, all(A2f.cocycle(q, p) == A2.cocycle(q, p) for q in vs for p in vs))
print(A2f.e((1, 1)).ope(A2f.e((-1, -1))))

False True
[-J0 - J1, -1]


## 4. Odd lattices: free fermions

If some $B_{ii}$ is odd, the $e^q$ with $B(q, q)$ odd are odd, and $V_L$ is a vertex superalgebra. The simplest case is $L = \mathbb Z$ with $B = (1)$. Then $b = e^{x_0}$ and $c = e^{-x_0}$ are odd, with $b(z)c(w) \sim 1/(z-w)$ and $c(z)b(w) \sim 1/(z-w)$, and ${:}bc{:} = J_0$. This is the boson–fermion correspondence [Kac1998]: the bc system of `04_ghost_systems.ipynb` is realised by a single free boson. The cell below prints the parities, the OPEs and the normally ordered products.

In [14]:
VZ = LatticeVertexAlgebra(QQ, matrix([[1]]))
b, c = VZ.e((1,)), VZ.e((-1,))
print('parities:', b.parity(), c.parity(), VZ.J(0).parity())
print('b c :', b.ope(c), '   c b :', c.ope(b), '   b b :', b.ope(b))
print(':b c: =', b.no(c), '   :c b: =', c.no(b))

parities: 1 1 0
b c : [1]    c b : [1]    b b : []
:b c: = J0    :c b: = -J0


In other words, $b \mapsto e^{x_0}$, $c \mapsto e^{-x_0}$ defines a morphism from the universal bc system `va.FermionicGhosts(QQ)` to $V_{\mathbb Z}$. `V.hom(images, W)` builds it and checks that the images satisfy the OPEs of the generators of `V`; since `V` is a universal vertex algebra this check is exact, and `checked_up_to_weight()` is `None` (see `04_ghost_systems.ipynb`, section 11).

In [15]:
phi = va.FermionicGhosts(QQ).hom([b, c], VZ)
print(phi)
print(phi.checked_up_to_weight())

Vertex algebra morphism:
  From: Universal vertex algebra on generators ['b', 'c'] over Rational Field
  To:   Lattice vertex algebra of rank 1 with Gram matrix [1] over Rational Field
  Defn: b |--> e^(1)
        c |--> e^(-1)
None


With $T_L = \frac12 {:}J_0J_0{:}$ the central charge is 1 and $e^{\pm x_0}$ have weight $\frac12$. The character of $V_{\mathbb Z}$ is then $\sum_{k} q^{k^2/2} / \prod_m (1-q^m)$, which by the Jacobi triple product equals the character $\prod_{n \ge 1} (1 + q^{n - 1/2})^2$ of a pair of free fermions of weight $\frac12$. The cell below prints the character and compares the dimensions of the weight spaces with the coefficients of the product, written in the variable $t = q^{1/2}$.

In [16]:
J = VZ.J(0)
VZs = VZ.with_conformal_vector(J.no(J) / 2)
print('c =', VZs.central_charge(), '  ', VZs.degrees())
print(VZs.character(4))
P.<t> = PowerSeriesRing(QQ, default_prec=8)          # t = q^(1/2)
print([VZs.weight_space(k / 2).dimension() for k in range(8)], list(prod((1 + t^(2*n - 1))^2 for n in range(1, 5)))[:8])

c = 1    {'J0': 1, 'e^(1)': 1/2, 'e^(-1)': 1/2}
1 + 2*q^(1/2) + q + 2*q^(3/2) + 4*q^2 + 4*q^(5/2) + 5*q^3 + 6*q^(7/2) + O(q^4)
[1, 2, 1, 2, 4, 4, 5, 6] [1, 2, 1, 2, 4, 4, 5, 6]


A background charge moves the weights. With $T = \frac12 {:}J_0J_0{:} - \frac12 \partial J_0$, $b$ has weight 1, $c$ has weight 0 and $c = -2$: these are the data of the bc system with `conformal_vector='b1c0'`, and the morphism from that algebra is conformal, that is, it maps conformal vector to conformal vector.

In [17]:
VZ10 = VZ.with_conformal_vector(J.no(J) / 2 - J.derivative() / 2)
print(VZ10.degrees(), '  c =', VZ10.central_charge())
phi10 = va.FermionicGhosts(QQ, conformal_vector='b1c0').hom([VZ10.e((1,)), VZ10.e((-1,))], VZ10)
print(phi10.is_conformal())

{'J0': 1, 'e^(1)': 1, 'e^(-1)': 0}   c = -2
True


## 5. Indefinite lattices: $I_{1,1}$

The Gram matrix need not be positive definite. $I_{1,1}$ has $B = \operatorname{diag}(1, -1)$: $x_0$ has norm $+1$ and $x_1$ has norm $-1$. Both $e^{\pm x_0}$ and $e^{\pm x_1}$ are odd.

There is no affine algebra here, but $T_L = \frac12{:}J_0J_0{:} - \frac12{:}J_1J_1{:}$ is still a conformal vector, of central charge 2, and $e^q$ has weight $B(q, q)/2 = (q_0^2 - q_1^2)/2$. This can be negative, and it vanishes on the null vectors $q_0 = \pm q_1$. The cell below prints the cocycle on the basis vectors, $T_L$ and $c$, and the parity and weight of $e^q$ for a few $q$.

In [18]:
I11 = LatticeVertexAlgebra(QQ, matrix([[1, 0], [0, -1]]))
print(I11)
print('ε(x0,x1) =', I11.cocycle((1, 0), (0, 1)), '   ε(x1,x0) =', I11.cocycle((0, 1), (1, 0)))
VI = I11.with_conformal_vector(lattice_conformal_vector(I11))
print('T =', VI.conformal_vector(), '   c =', VI.central_charge())
for q in [(1, 0), (0, 1), (1, 1), (1, -1), (2, 1), (1, 2)]:
    print('q = %-8s parity %d   weight %s' % (q, VI.e(q).parity(), VI.e(q).degree()))

Lattice vertex algebra of rank 2 with Gram matrix [[1, 0], [0, -1]] over Rational Field
ε(x0,x1) = 1    ε(x1,x0) = -1
T = 1/2*:J0 J0: - 1/2*:J1 J1:    c = 2
q = (1, 0)   parity 1   weight 1/2
q = (0, 1)   parity 1   weight -1/2
q = (1, 1)   parity 0   weight 0
q = (1, -1)  parity 0   weight 0
q = (2, 1)   parity 1   weight 3/2
q = (1, 2)   parity 1   weight -3/2


**Weight spaces.** Since $e^{k(x_0 + x_1)}$ has weight 0 for every $k$, every weight space of $V_{I_{1,1}}$ is infinite-dimensional, and the character does not exist. The remedy, as for the βγ system of `04_ghost_systems.ipynb` (section 2), is a second grading: `with_horizontal_grading('lattice')` grades $V_L$ by the lattice, $e^q$ and every monomial ${:}\cdots e^q{:}$ having grade $q$ and the currents grade 0. The space of a given weight *and* grade is finite-dimensional; it is `V.weight_space(n, horizontal_grade=q)`.

The cell below shows that the weight-0 space is infinite-dimensional and that the character is refused, and then the dimensions of the spaces of grade $(1, 1)$. Since $e^{(1,1)}$ has weight 0, this graded piece is $e^{(1,1)}$ times the Fock space of two bosons, of dimensions $1, 2, 5, 10, 20, 36, \dots$ (the coefficients of $\prod_m (1-q^m)^{-2}$).

In [19]:
print(VI.weight_space(0).dimension())
try:
    VI.character(3)
except ValueError as err:
    print('ValueError:', err)
VIg = VI.with_horizontal_grading('lattice')
print(VIg.e((2, 1)).horizontal_grade(), VIg.J(0).horizontal_grade())
print([VIg.weight_space(n, horizontal_grade=(1, 1)).dimension() for n in range(6)])
print(VIg.weight_space(1, horizontal_grade=(1, 1)).basis())

+Infinity
ValueError: the weight spaces are not certified to be finite dimensional: give a grade
(2, 1) (0, 0)
[1, 2, 5, 10, 20, 36]
[:J0 e^(1,1):, :J1 e^(1,1):]


**Signs in the negative direction.** Since $B(x_1, x_1) = -1$, the OPEs of $u = e^{x_1}$ and $v = e^{-x_1}$ behave the opposite way to those of $e^{\pm x_0}$: $u(z)v(w)$ starts at $(z-w)^{+1}$, so it is regular and even ${:}uv{:} = 0$ (the first nonzero product is $u_{(-2)}v$), while $u(z)u(w)$ starts at $(z-w)^{-1}$ and has a pole.

The isotropic vectors $n_\pm = e^{\pm(x_0 + x_1)}$ are even, of weight 0, and $B(x_0 + x_1, -x_0 - x_1) = 0$: their OPE is regular, and their normally ordered product is $\varepsilon(x_0 + x_1, -x_0 - x_1)\,\mathbf 1 = -\mathbf 1$, a sign from the cocycle.

In [20]:
u, v = I11.e((0, 1)), I11.e((0, -1))
print('u v :', u.ope(v), '   :u v: =', u.no(v), '   u_(-2) v =', u.nproduct(v, -2))
print('u u :', u.ope(u))
n_plus, n_minus = I11.e((1, 1)), I11.e((-1, -1))
print('n+ n- :', n_plus.ope(n_minus), '   :n+ n-: =', n_plus.no(n_minus), '   ε =', I11.cocycle((1, 1), (-1, -1)))

u v : []    :u v: = 0    u_(-2) v = 1
u u : [e^(0,2)]
n+ n- : []    :n+ n-: = -1    ε = -1


## 6. The βγ system inside $V_{I_{1,1}}$

Friedan, Martinec and Shenker realise the βγ system by free fields [FMS1986]. In the language of lattice vertex algebras, the fields

$$\beta = e^{x_0 + x_1}, \qquad \gamma = {:}J_0\, e^{-x_0 - x_1}{:}$$

of $V_{I_{1,1}}$ are even (the lattice vectors are null) and satisfy $\beta(z)\gamma(w) \sim 1/(z-w)$, with regular OPEs $\beta(z)\beta(w)$ and $\gamma(z)\gamma(w)$. These are the OPEs of the βγ system of `04_ghost_systems.ipynb`. The coefficient ring is $\mathbb Q(\lambda)$ (the variable `lam`), for the conformal vector below.

In [21]:
Kl.<lam> = Frac(QQ['lam'])
V = LatticeVertexAlgebra(Kl, matrix([[1, 0], [0, -1]]))
J0, J1 = V.heisenberg_currents()
beta, gamma = V.e((1, 1)), J0.no(V.e((-1, -1)))
print('parities:', beta.parity(), gamma.parity())
print('β γ :', beta.ope(gamma), '   γ β :', gamma.ope(beta), '   β β :', beta.ope(beta), '   γ γ :', gamma.ope(gamma))

parities: 0 0
β γ : [1]    γ β : [-1]    β β : []    γ γ : []


The βγ system has the conformal vectors $T_\lambda$ for which $\beta$ and $\gamma$ are primary of weights $\lambda$ and $1 - \lambda$ (`04_ghost_systems.ipynb`, section 1). Here they come from a background charge, $T = T_L + a_0\, \partial J_0 + a_1\, \partial J_1$:

* $\beta = e^{x_0 + x_1}$ is primary for every $a_0, a_1$, of weight $a_1 - a_0$;
* $\gamma$ is primary only if $a_0 = -\frac12$, since $T_{(2)}\gamma = -(1 + 2a_0)\, e^{-x_0-x_1}$.

So $a_0 = -\frac12$ and $a_1 = \lambda - \frac12$. The central charge is then $2 - 12(a_0^2 - a_1^2) = 2(6\lambda^2 - 6\lambda + 1)$, that of the βγ system of weight $\lambda$. The cell below builds $T$, prints $c$ and the weights of $\beta$ and $\gamma$, and checks that both are primary: $T(z)\beta(w) \sim \frac{\lambda \beta(w)}{(z-w)^2} + \frac{\partial\beta(w)}{z-w}$, and likewise for $\gamma$ with weight $1 - \lambda$.

In [22]:
VT = V.with_conformal_vector(lattice_conformal_vector(V) - J0.derivative() / 2 + (lam - 1/2) * J1.derivative())
T = VT.conformal_vector()
beta, gamma = VT.e((1, 1)), VT.J(0).no(VT.e((-1, -1)))
print('c =', VT.central_charge())
print('weights:', beta.degree(), gamma.degree())
print('primary:', T.ope(beta) == [beta.derivative(), lam * beta], T.ope(gamma) == [gamma.derivative(), (1 - lam) * gamma])

c = 12*lam^2 - 12*lam + 2
weights: lam -lam + 1
primary: True True


So $\beta \mapsto e^{x_0+x_1}$, $\gamma \mapsto {:}J_0e^{-x_0-x_1}{:}$ defines a morphism from the βγ system `va.BosonicGhosts(Kl, beta_weight=lam)` to $V_{I_{1,1}}$, and it maps $T_\lambda$ to $T$. The cell below builds it; the check of the OPEs is exact, and the morphism is conformal.

In [23]:
Bg = va.BosonicGhosts(Kl, beta_weight=lam)
phi = Bg.hom([beta, gamma], VT)
print(phi)
print(phi.checked_up_to_weight(), phi.is_conformal(), phi(Bg.conformal_vector()) == T)

Vertex algebra morphism:
  From: Universal vertex algebra on generators ['beta', 'gamma'] over Fraction Field of Univariate Polynomial Ring in lam over Rational Field
  To:   Lattice vertex algebra of rank 2 with Gram matrix [[1, 0], [0, -1]] over Fraction Field of Univariate Polynomial Ring in lam over Rational Field
  Defn: beta |--> e^(1,1)
        gamma |--> :J0 e^(-1,-1):
None True True


The βγ system is simple, so the morphism is injective. For $\lambda = \frac12$ (over $\mathbb Q$) the cell below computes its kernel and image weight space by weight space (`phi.kernel()`, `phi.image()`, see `10_modules.ipynb`, section 9). The kernel is zero and the image has the dimensions of the βγ system, $1, 2, 3, 6, 9, 14, \dots$ (`04_ghost_systems.ipynb`, section 2), in each weight computed. The image is a small part of $V_{I_{1,1}}$, whose weight spaces are infinite-dimensional (section 5).

In [24]:
VTh = I11.with_conformal_vector(lattice_conformal_vector(I11) - I11.J(0).derivative() / 2)
Bh = va.BosonicGhosts(QQ, conformal_vector='symmetric')
phih = Bh.hom([VTh.e((1, 1)), VTh.J(0).no(VTh.e((-1, -1)))], VTh)
Ker, Im = phih.kernel(), phih.image()
print(phih.is_conformal())
print([Ker.weight_space(k / 2).dimension() for k in range(6)])
print([Im.weight_space(k / 2).dimension() for k in range(6)], [Bh.weight_space(k / 2).dimension() for k in range(6)])
print(VTh.weight_space(1/2).dimension())

True
[0, 0, 0, 0, 0, 0]
[1, 2, 3, 6, 9, 14] [1, 2, 3, 6, 9, 14]
+Infinity


## 7. Tensor products

`V.tensor(W)` (or `TensorProductVertexAlgebra([V, W, ...])`) is the tensor product of vertex (super)algebras over the same ring, with $Y(a \otimes b, z) = Y(a, z) \otimes Y(b, z)$. For superalgebras it includes the Koszul sign: odd elements of different factors anticommute. With `conformal_vector='factors'` its conformal vector is the sum of those of the factors, and the central charges add (as in `04_ghost_systems.ipynb`, section 9). `embed(i, x)` puts `x` into the $i$-th factor, `pure_tensor(x, y, ...)` is $x \otimes y \otimes \cdots$, and the generators are those of the factors, with the index of the factor appended to their names when the names clash.

**$V_{\mathbb Z} \otimes V_{\mathbb Z} \cong V_{\mathbb Z^2}$.** The orthogonal sum of lattices gives the tensor product of their vertex algebras. With odd lattices there is a sign to get right: in $V_{\mathbb Z} \otimes V_{\mathbb Z}$ the fermions $\psi_0 = e^{x_0} \otimes \mathbf 1$ and $\psi_1 = \mathbf 1 \otimes e^{x_0}$ anticommute because of the Koszul sign, while in $V_{\mathbb Z^2}$, with $B = \operatorname{diag}(1, 1)$, $e^{x_0}$ and $e^{x_1}$ anticommute because of the cocycle, $\varepsilon(x_1, x_0) = -1$. The cell below prints the normally ordered products in both.

In [25]:
VZ2 = VZs.tensor(VZs, conformal_vector='factors')
print(VZ2)
print(VZ2.variable_names(), '  c =', VZ2.central_charge())
psi0, psi1 = VZ2.embed(0, VZs.e((1,))), VZ2.embed(1, VZs.e((1,)))
print('V_Z ⊗ V_Z :', psi0.ope(psi1), '   :psi0 psi1: =', psi0.no(psi1), '   :psi1 psi0: =', psi1.no(psi0))
I2 = LatticeVertexAlgebra(QQ, matrix.identity(2))
VI2 = I2.with_conformal_vector(lattice_conformal_vector(I2))
f0, f1 = VI2.e((1, 0)), VI2.e((0, 1))
print('V_(Z^2)   :', f0.ope(f1), '   :f0 f1: =', f0.no(f1), '   :f1 f0: =', f1.no(f0))

Tensor product of vertex algebras [Lattice vertex algebra of rank 1 with Gram matrix [1] over Rational Field, Lattice vertex algebra of rank 1 with Gram matrix [1] over Rational Field]
('J0_0', 'e^(1)_0', 'e^(-1)_0', 'J0_1', 'e^(1)_1', 'e^(-1)_1')   c = 2
V_Z ⊗ V_Z : []    :psi0 psi1: = e^(1) # e^(1)    :psi1 psi0: = -e^(1) # e^(1)
V_(Z^2)   : []    :f0 f1: = e^(1,1)    :f1 f0: = -e^(1,1)


The isomorphism sends the generators of the first factor to $J_0, e^{\pm x_0}$ and those of the second to $J_1, e^{\pm x_1}$. The check of the images is exact here too (`checked_up_to_weight()` is `None`). The cell below builds the morphism, checks that it is conformal, and compares kernel, image and $V_{\mathbb Z^2}$ in the weights $0, \frac12, \dots, 3$.

In [26]:
iso = VZ2.hom([VI2.J(0), VI2.e((1, 0)), VI2.e((-1, 0)), VI2.J(1), VI2.e((0, 1)), VI2.e((0, -1))], VI2)
print(iso.checked_up_to_weight(), iso.is_conformal())
Ker, Im = iso.kernel(), iso.image()
print([Ker.weight_space(k / 2).dimension() for k in range(7)])
print([Im.weight_space(k / 2).dimension() for k in range(7)], [VI2.weight_space(k / 2).dimension() for k in range(7)])

None True
[0, 0, 0, 0, 0, 0, 0]
[1, 4, 6, 8, 17, 28, 38] [1, 4, 6, 8, 17, 28, 38]


**A universal algebra times a lattice algebra.** The factors can be any vertex algebras over the same ring. Below, the Neveu–Schwarz algebra of central charge $c_0$ (a universal vertex algebra with an odd generator $G$ of weight $\frac32$, from the catalog) times $V_{I_{1,1}}$ with $T_L$. The central charge is $c_0 + 2$, and the odd fields $G$ and $e^{x_0}$ of different factors anticommute. The last line shows the OPE of the total conformal vector with $G \otimes e^{x_0}$, of weight $\frac32 + \frac12 = 2$.

In [27]:
Kc.<c0> = Frac(QQ['c0'])
NS = va.NeveuSchwarz(Kc, c0, conformal_vector='L')
IK = LatticeVertexAlgebra(Kc, matrix([[1, 0], [0, -1]]))
IK = IK.with_conformal_vector(lattice_conformal_vector(IK))
M = NS.tensor(IK, conformal_vector='factors')
print(M.variable_names())
print('c =', M.central_charge())
G, ex0 = M.embed(0, NS.gen('G')), M.embed(1, IK.e((1, 0)))
print('parities:', G.parity(), ex0.parity(), '   :G e^x0: == -:e^x0 G: :', G.no(ex0) == -ex0.no(G))
x = M.pure_tensor(NS.gen('G'), IK.e((1, 0)))
print(x, '  weight', x.degree(), '  T x ~', M.conformal_vector().ope(x))

('L', 'G', 'J0', 'J1', 'e^(1,0)', 'e^(0,1)', 'e^(-1,0)', 'e^(0,-1)')
c = c0 + 2
parities: 1 1    :G e^x0: == -:e^x0 G: : True
G # e^(1,0)   weight 2   T x ~ [G # :J0 e^(1,0): + dG # e^(1,0), 2*G # e^(1,0)]


**Heisenberg times $V_{A_1}$.** The universal Heisenberg algebra on $J$, with $J(z)J(w) \sim 1/(z-w)^2$, tensored with $V_{A_1}$, maps to $V_L$ for $B = \operatorname{diag}(1, 2)$ by $J \mapsto J_0$ and $J_0, e^{\pm x_0} \mapsto J_1, e^{\pm x_1}$. The map is injective, but it is not onto: its image is the part of $V_L$ spanned by the monomials ${:}\cdots e^q{:}$ with $q_0 = 0$, since $e^{\pm x_0}$ are not in it. ($V_L$ itself is $V_{\mathbb Z} \otimes V_{A_1}$.) The cell below builds the morphism and compares the dimensions of its kernel, its image and $V_L$.

In [28]:
Hu = UniversalVertexAlgebra(QQ, ['J'], {('J', 'J'): [0, 1]})
Hs = Hu.with_conformal_vector(Hu.gen('J').no(Hu.gen('J')) / 2)
HA = Hs.tensor(VA1, conformal_vector='factors')
L12 = LatticeVertexAlgebra(QQ, matrix([[1, 0], [0, 2]]))
L12 = L12.with_conformal_vector(lattice_conformal_vector(L12))
psi = HA.hom([L12.J(0), L12.J(1), L12.e((0, 1)), L12.e((0, -1))], L12)
print(psi)
print(psi.checked_up_to_weight(), psi.is_conformal())
Ker, Im = psi.kernel(), psi.image()
print([Ker.weight_space(n).dimension() for n in range(6)])
print([Im.weight_space(n).dimension() for n in range(6)], [L12.weight_space(n).dimension() for n in range(6)])

Vertex algebra morphism:
  From: Tensor product of vertex algebras [Universal vertex algebra on generators ['J'] over Rational Field, Lattice vertex algebra of rank 1 with Gram matrix [2] over Rational Field]
  To:   Lattice vertex algebra of rank 2 with Gram matrix [[1, 0], [0, 2]] over Rational Field
  Defn: J |--> J0
        J0 |--> J1
        e^(1) |--> e^(0,1)
        e^(-1) |--> e^(0,-1)
None True
[0, 0, 0, 0, 0, 0]
[1, 4, 9, 20, 42, 80] [1, 4, 11, 28, 60, 120]


## 8. Unique representation, pickling and Sage lattices

Lattice vertex algebras have unique representation: equal data give the same object, however they are described. The Gram matrix can be given as a Sage matrix, as a `CartanMatrix`, or as a Sage `IntegralLattice`, whose Gram matrix is then used; `va.Lattice` is the same constructor. The conformal vector is part of the data, so `with_conformal_vector` gives the same object as `conformal_vector=` in the constructor.

The algebras can be saved and loaded (`dumps`, `loads`), which returns the same object, and they pass Sage's generic tests (`TestSuite`), which include the vertex algebra axioms on sample elements. A cocycle given as a function is saved by its name, so it must be defined when the algebra is loaded (section 9).

In [29]:
print(va.Lattice(QQ, matrix([[2, -1], [-1, 2]])) is A2,
      LatticeVertexAlgebra(QQ, CartanMatrix(['A', 2])) is A2,
      LatticeVertexAlgebra(QQ, IntegralLattice('A2')) is A2)
print(LatticeVertexAlgebra(QQ, matrix([[2, -1], [-1, 2]]), conformal_vector=lattice_conformal_vector(A2)) is VA2)
print(loads(dumps(VA2)) is VA2, loads(dumps(VI)) is VI, loads(dumps(A2f)) is A2f)
TestSuite(VA2).run(); TestSuite(VI).run(); TestSuite(VZs).run()
print('TestSuite ok')

True True True
True
True True True
TestSuite ok


## 9. Limitations

* **The form.** The Gram matrix must be symmetric, integral and non-degenerate, and the coefficient ring must contain $\mathbb Q$. A rational, non-integral form, such as $B = (\frac12)$, defines a generalised vertex algebra [DL1993], which is not constructed. Vectors of the dual lattice $L^\circ$ appear only through the simple modules $V_{\lambda + L}$ (`10_modules.ipynb`, section 2).
* **Conformal vectors have no names.** Affine vertex algebras and ghost systems take a name such as `conformal_vector='sugawara'`; a lattice vertex algebra takes none (a name is refused with a `TypeError`), even for a root lattice, where $T_L$ is the Sugawara vector of the affine algebra it realises (sections 2 and 3). The conformal vector is always written as an element, here with `lattice_conformal_vector`.
* **Cocycles given as functions are not checked.** Neither the cocycle identity, nor the condition $\varepsilon(q,p)/\varepsilon(p,q) = (-1)^{B(q,p) + B(q,q)B(p,p)}$, nor $\varepsilon(0, p) = \varepsilon(q, 0) = 1$ is tested, and a wrong function gives an algebra that silently violates the axioms. Check one with `TestSuite` or with a test such as `skew_symmetric` in section 3. A function cocycle is pickled by its name, so a `lambda` cannot be saved, and a function defined in a notebook must be defined again before loading.
* **Indefinite lattices.** The weight spaces are infinite-dimensional (section 5): characters and bases need a grade (`horizontal_grade=`), and the Zhu algebra of a lattice that is not positive definite is zero (`09_zhu_algebras.ipynb`, section 8).
* **Generators.** $J_i$ and $e^{\pm x_i}$ generate $V_L$ but the library knows no finite set of strong generators for it. Constructions that need one are not available for lattice vertex algebras: the Nahm–Gaberdiel–Kausch fusion algorithm is replaced by the known fusion rules $V_{\lambda+L} \boxtimes V_{\mu+L} = V_{\lambda+\mu+L}$ [DL1993] (`11_fusion_and_extensions.ipynb`). Morphisms *out of* $V_L$ are checked by conditions on the images of $e^{\pm x_i}$ alone, by the universal property of lattice vertex algebras [LL2004, §6.5].
* **Modules.** The simple modules $V_{\lambda+L}$, $\lambda \in L^\circ$, are constructed [Don1993, DLM1997], and from them pullbacks and spectral flows (`10_modules.ipynb`). There is no constructor for $g$-twisted modules, for example those twisted by the lift of $-1$ on $L$.
* **Cost.** $e^q_{(n)}e^p$ involves the complete symmetric functions of degree up to $-n-1-B(q,p)$, so products of lattice currents with very negative $B(q, p)$, or with many Heisenberg letters, grow quickly. Products run in C++ over $\mathbb Q$, number fields, polynomial rings over these and their fraction fields, and in Python over other rings; threads are set as in `01_getting_started.ipynb`, section 8.

The cell below shows the refusals of the first two items: a coefficient ring without $\mathbb Q$, a non-integral and a degenerate Gram matrix, and a conformal vector given by name.

In [30]:
for R0, B0 in [(ZZ, matrix([[2]])), (QQ, matrix([[1/2]])), (QQ, matrix([[1, 1], [1, 1]]))]:
    try:
        LatticeVertexAlgebra(R0, B0)
    except ValueError as err:
        print('ValueError:', err)
try:
    LatticeVertexAlgebra(QQ, matrix([[2]]), conformal_vector='sugawara')
except TypeError as err:
    print('TypeError:', err)

ValueError: the base ring must contain QQ
ValueError: the Gram matrix must be integral
ValueError: the Gram matrix must be non-degenerate
TypeError: the conformal vector must be an element of the lattice vertex algebra (or a dict of its basis keys), not 'sugawara'


## References

- [Bor1986] R. E. Borcherds, *Vertex algebras, Kac–Moody algebras, and the Monster*, Proc. Nat. Acad. Sci. U.S.A. 83 (1986), 3068–3071, [doi:10.1073/pnas.83.10.3068](https://doi.org/10.1073/pnas.83.10.3068). — lattice vertex algebras.
- [Kac1998] V. G. Kac, *Vertex algebras for beginners*, 2nd ed., Univ. Lecture Ser. 10, AMS (1998), [doi:10.1090/ulect/010](https://doi.org/10.1090/ulect/010). — the construction, conventions and cocycles (§5.4–5.5; sections 2, 3), the boson–fermion correspondence (section 4).
- [FK1980] I. B. Frenkel, V. G. Kac, *Basic representations of affine Lie algebras and dual resonance models*, Invent. Math. 62 (1980), 23–66, [doi:10.1007/BF01391662](https://doi.org/10.1007/BF01391662). — affine algebras at level 1 from root lattices (sections 1, 3).
- [Seg1981] G. Segal, *Unitary representations of some infinite dimensional groups*, Comm. Math. Phys. 80 (1981), 301–342, [doi:10.1007/BF01208274](https://doi.org/10.1007/BF01208274). — affine algebras at level 1 from root lattices (section 1).
- [FMS1986] D. Friedan, E. Martinec, S. Shenker, *Conformal invariance, supersymmetry and string theory*, Nuclear Phys. B 271 (1986), 93–165, [doi:10.1016/0550-3213(86)90356-1](https://doi.org/10.1016/0550-3213(86)90356-1). — the bosonization of the βγ system (section 6).
- [DL1993] C. Dong, J. Lepowsky, *Generalized vertex algebras and relative vertex operators*, Progr. Math. 112, Birkhäuser (1993), [doi:10.1007/978-1-4612-0353-7](https://doi.org/10.1007/978-1-4612-0353-7). — generalised vertex algebras of rational lattices, fusion rules of lattice vertex algebras (section 9).
- [LL2004] J. Lepowsky, H. Li, *Introduction to vertex operator algebras and their representations*, Progr. Math. 227, Birkhäuser (2004), [doi:10.1007/978-0-8176-8186-9](https://doi.org/10.1007/978-0-8176-8186-9). — the universal property of lattice vertex algebras (§6.5; section 9).
- [Don1993] C. Dong, *Vertex algebras associated with even lattices*, J. Algebra 161 (1993), 245–265, [doi:10.1006/jabr.1993.1217](https://doi.org/10.1006/jabr.1993.1217). — the simple modules of lattice vertex algebras (section 9).
- [DLM1997] C. Dong, H. Li, G. Mason, *Regularity of rational vertex operator algebras*, Adv. Math. 132 (1997), 148–166, [arXiv:q-alg/9508018](https://arxiv.org/abs/q-alg/9508018). — the modules of lattice vertex algebras (section 9).